# Long-Term Health Condition Analysis & Risk Identifier: Notebook Dashboard

**Master of Data Science project. Data: AIHW / ABS Census 2021 (Tables 10, 13 & 14)**

This notebook builds the whole project from scratch, with **Plotly Express** used for every chart:

| Section | What it does |
|---|---|
| 1. Setup | Imports and settings |
| 2. Data preparation | Loads and cleans the data and explains its structure |
| 3. Analytics | Prevalence, age-standardisation and elevated-risk groups |
| 4. Charts | One Plotly Express function per chart, previewed in the notebook |
| 5. Analysis dashboard | Five tabs with global filters, **displayed inside the notebook** |
| 6. Risk Identifier | A form where the user describes a person and sees their likely conditions |

**Before you start:** put `Cleaned_data_final.xlsx` in the same folder as this notebook (or in a `data` sub-folder), then run the cells from top to bottom.

Everything else, including the dashboard styling, is inside this notebook. No other `.py` or `.css` files are needed.

## 1. Setup

In [ ]:
# Install the packages into THIS notebook's Python (safe to re-run; quick if already installed)
%pip install dash plotly pandas numpy openpyxl

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from dash import Dash, Input, Output, State, dash_table, dcc, html

## 2. Data preparation

The cleaned file stacks **two census cross-tabulations**:

| View | Rows where | Breakdown |
|---|---|---|
| **Migration** (Table 10) | `years` is specified | country × years in Australia × age × sex × condition |
| **Language** (Tables 13/14) | `proficiency` is specified | country × language at home × English proficiency × age × sex × condition |

Both views describe the **same people**, so they are never added together. Every chart uses exactly one of them.

### 2.1 File location, column names and category lists

In [ ]:
# Look for the data file next to the notebook or in a "data" folder
CANDIDATES = [Path("Cleaned_data_final.xlsx"), Path("data/Cleaned_data_final.xlsx"),
              Path("Cleaned_data_final.csv.xlsx"), Path("data/Cleaned_data_final.csv.xlsx")]
RAW_FILE = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
CACHE_FILE = RAW_FILE.with_name("lthc_clean.csv.gz")   # fast-loading copy made on first run
print("Data file:", RAW_FILE.resolve())

COLUMN_MAP = {
    "Country of birth of person": "country",
    "Years spent in Australia": "years",
    "Age group": "age",
    "Sex": "sex",
    "Long-term health condition (LTHC)": "condition",
    "Number of people reporting LTHC(s)": "cases",
    "Population": "population",
    "Age-specific percentage of population reporting LTHC(s)": "rate",
    "Language used at home": "language",
    "Proficiency in spoken English": "proficiency",
    "Region_class": "region",
    "Subregion_class": "subregion",
}

NS = "Not specified"
ANY_LTHC = "One or more long-term health condition(s)"
AGE_ORDER = ["0–44", "45–64", "65 and over"]
SEX_ORDER = ["Persons", "Male", "Female"]
YEARS_ORDER = ["0–10 years", "More than 10 years"]
PROF_ORDER = ["Very well or well", "Not well or Not at all"]

# Individual conditions (excludes the roll-up categories so they can be compared)
SPECIFIC_CONDITIONS = [
    "Arthritis", "Asthma", "Cancer", "Dementia", "Diabetes", "Heart disease",
    "Kidney disease", "Lung condition", "Mental health condition", "Stroke",
]
SUMMARY_CONDITIONS = [
    ANY_LTHC, "Heart disease or stroke", "Any other long-term health condition(s)",
]
ALL_CONDITIONS = SUMMARY_CONDITIONS[:1] + SPECIFIC_CONDITIONS + SUMMARY_CONDITIONS[1:]

# SACC country names -> names recognised by Plotly's "country names" location mode
MAP_NAME_FIX = {
    "China (excludes SARs and Taiwan)": "China",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Wales": "United Kingdom",
    "Northern Ireland": "United Kingdom",
    "United Kingdom, Channel Islands and Isle of Ma": "United Kingdom",
    "Hong Kong (SAR of China)": "Hong Kong",
    "Macau (SAR of China)": "Macao",
    "Korea, Republic of (South)": "South Korea",
    "Korea, Democratic People's Republic of (North)": "North Korea",
    "Congo, Democratic Republic of": "Democratic Republic of the Congo",
    "Congo, Republic of": "Republic of the Congo",
    "Russian Federation": "Russia",
    "Gaza Strip and West Bank": "Palestine",
    "Micronesia, Federated States of": "Micronesia",
    "Samoa, American": "American Samoa",
    "Eswatini": "Swaziland",
    "Cabo Verde": "Cape Verde",
    "Brunei Darussalam": "Brunei",
    "Czechia": "Czech Republic",
    "Timor-Leste": "East Timor",
    "North Macedonia": "Macedonia",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St Lucia": "Saint Lucia",
    "St Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Virgin Islands, British": "British Virgin Islands",
    "Virgin Islands, United States": "United States Virgin Islands",
    "Cote d'Ivoire": "Ivory Coast",
}

### 2.2 Load and clean
The loader does the following:

- Fixes garbled dashes (`â€“` → `–`) and renames the columns.
- Treats rows with a population of 0 as **unpublished** cells, not as zero cases.
- Recalculates each rate as cases ÷ population.

In [ ]:
def _fix_text(s: pd.Series) -> pd.Series:
    """Repair UTF-8 read as cp1252 (e.g. 'â€“' -> '–') and trim whitespace."""
    return (
        s.astype(str)
        .str.replace("â€“", "–", regex=False)
        .str.replace("â€”", "—", regex=False)
        .str.replace("â€™", "’", regex=False)
        .str.strip()
    )


def load_data(raw_path: Path | str = RAW_FILE, use_cache: bool = True) -> pd.DataFrame:
    """Load the cleaned census extract and return a tidy DataFrame."""
    if use_cache and CACHE_FILE.exists():
        return pd.read_csv(CACHE_FILE)

    raw_path = Path(raw_path)
    if raw_path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(raw_path)
    else:
        df = pd.read_csv(raw_path, encoding="utf-8")

    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])
    df = df.rename(columns=COLUMN_MAP)

    for col in ["country", "years", "age", "sex", "condition", "language",
                "proficiency", "region", "subregion"]:
        df[col] = _fix_text(df[col])
    df["age"] = df["age"].replace({"00–44": "0–44"})
    df["language"] = df["language"].replace({".ATSI lang": "Aboriginal & Torres Strait Islander languages"})

    df["cases"] = pd.to_numeric(df["cases"], errors="coerce").fillna(0).astype(int)
    df["population"] = pd.to_numeric(df["population"], errors="coerce").fillna(0).astype(int)
    df["view"] = np.where(df["years"] != NS, "migration", "language")

    # Rows with population = 0 are cells that were NOT published for that
    # condition (e.g. sex-specific figures for some conditions / countries).
    # They are "missing", not "zero cases": because both cases and population
    # are 0 they drop out of every sum(cases)/sum(population) automatically.
    # ``group_population`` keeps the full size of each demographic cell
    # (same for every condition) for display purposes.
    df["available"] = df["population"] > 0
    key_lang = np.where(df["view"] == "language", df["language"], "")
    df["group_population"] = (df.assign(_k=key_lang)
                              .groupby(["view", "country", "years", "_k", "proficiency", "age", "sex"])
                              ["population"].transform("max"))
    df["rate"] = np.where(df["population"] > 0, df["cases"] / df["population"] * 100, 0.0)

    df["map_country"] = df["country"].replace(MAP_NAME_FIX)

    if use_cache:
        try:
            df.to_csv(CACHE_FILE, index=False, compression="gzip")
        except OSError:
            pass
    return df


def split_views(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (migration_df, language_df). See module docstring."""
    mig = df[df["view"] == "migration"].copy()
    lang = df[df["view"] == "language"].copy()
    return mig, lang

In [ ]:
# Load the data once; everything below reuses these tables
DF = load_data()
MIG, LANG = split_views(DF)   # migration view / language view
print(f"Rows: {len(DF):,}  (migration view {len(MIG):,}, language view {len(LANG):,})")
DF.head()

## 3. Analytics

- **Prevalence** = sum(cases) ÷ sum(population) after grouping. Percentages are never averaged.
- **Age-standardised prevalence (ASR)** reweights each group to the same age mix, so groups with very different age profiles can be compared fairly.
- **Risk table** compares each group with the same age group using a binomial z-score and a rate ratio.

In [ ]:
def prevalence(df: pd.DataFrame, by: list[str] | str | None = None) -> pd.DataFrame:
    """Aggregate cases & population and recompute prevalence (%)."""
    if by is None or by == []:
        c, p = df["cases"].sum(), df["population"].sum()
        return pd.DataFrame({"cases": [c], "population": [p],
                             "rate": [c / p * 100 if p else np.nan]})
    by = [by] if isinstance(by, str) else list(by)
    out = df.groupby(by, observed=True, as_index=False)[["cases", "population"]].sum()
    out["rate"] = np.where(out["population"] > 0, out["cases"] / out["population"] * 100, np.nan)
    return out


def standard_weights(mig: pd.DataFrame) -> dict[str, float]:
    """Age distribution of the whole overseas-born population (standard population)."""
    base = mig[(mig["sex"] == "Persons") & (mig["condition"] == ANY_LTHC)]
    w = base.groupby("age")["population"].sum()
    return (w / w.sum()).to_dict()


def age_standardised(df: pd.DataFrame, by: list[str] | str, weights: dict[str, float]) -> pd.DataFrame:
    """
    Directly age-standardised prevalence (%) for each group in ``by``.

    Groups differ a lot in age structure (e.g. recent arrivals are young), so
    comparing crude rates is misleading. Each group's age-specific rates are
    weighted by the standard population. Age bands with no population in a
    group are dropped and the remaining weights re-normalised.
    """
    by = [by] if isinstance(by, str) else list(by)
    g = prevalence(df, by + ["age"])
    g = g[g["population"] > 0].copy()
    g["w"] = g["age"].map(weights).fillna(0)
    g["wr"] = g["rate"] * g["w"]
    out = g.groupby(by, as_index=False).agg(
        wr=("wr", "sum"), w=("w", "sum"), cases=("cases", "sum"), population=("population", "sum"))
    out["asr"] = np.where(out["w"] > 0, out["wr"] / out["w"], np.nan)
    out["crude_rate"] = out["cases"] / out["population"] * 100
    return out.drop(columns=["wr", "w"])


def risk_table(df: pd.DataFrame, group_cols: list[str], baseline_cols: list[str],
               min_population: int = 500) -> pd.DataFrame:
    """
    Compare each group's prevalence with a baseline group (e.g. same age & sex,
    all countries) and flag statistically elevated risk.

    * rate_ratio = group rate / baseline rate
    * z          = binomial z-score of the observed rate vs the baseline rate
    * risk_level = "Elevated"  if z >= 3 and ratio >= 1.2
                   "Lower"     if z <= -3 and ratio <= 0.8
                   "Typical"   otherwise
    """
    grp = prevalence(df, group_cols)
    base = prevalence(df, baseline_cols).rename(
        columns={"rate": "baseline_rate", "cases": "_bc", "population": "_bp"})
    out = grp.merge(base[baseline_cols + ["baseline_rate"]], on=baseline_cols, how="left")
    out = out[out["population"] >= min_population].copy()

    p0 = out["baseline_rate"] / 100
    se = np.sqrt(p0 * (1 - p0) / out["population"])
    out["z_score"] = np.where(se > 0, (out["rate"] / 100 - p0) / se, 0.0)
    out["rate_ratio"] = np.where(out["baseline_rate"] > 0, out["rate"] / out["baseline_rate"], np.nan)
    out["risk_level"] = np.select(
        [(out["z_score"] >= 3) & (out["rate_ratio"] >= 1.2),
         (out["z_score"] <= -3) & (out["rate_ratio"] <= 0.8)],
        ["Elevated", "Lower"], default="Typical")
    return out.sort_values("z_score", ascending=False)


def fmt_int(n: float) -> str:
    return f"{int(round(n)):,}" if pd.notna(n) else "–"


def fmt_pct(x: float, digits: int = 1) -> str:
    return f"{x:.{digits}f}%" if pd.notna(x) else "–"

In [ ]:
# Standard age weights and quick summary
STD_W = standard_weights(MIG)
tot = prevalence(MIG[(MIG.sex == "Persons") & (MIG.condition == ANY_LTHC)])
print(f"Overseas-born population covered: {tot.population[0]:,}")
print(f"Reporting 1+ long-term condition: {tot.cases[0]:,} ({tot.rate[0]:.1f}%)")
prevalence(MIG[MIG.sex == "Persons"], "condition").sort_values("rate", ascending=False).round(2)

## 4. Charts (Plotly Express)

Each chart is a function that takes the dashboard filters and returns a Plotly Express figure. The same functions are previewed here and then reused by the dashboard in Section 5.

### 4.1 Theme, colours and small helpers

In [ ]:
NAVY, TEAL, CORAL, AMBER, GREY = "#0b2545", "#0f9d8f", "#e4572e", "#f2a541", "#8a96a8"
SEX_COLORS = {"Male": "#3d6fb6", "Female": "#d1495b", "Persons": NAVY}
YEARS_COLORS = {"0–10 years": AMBER, "More than 10 years": NAVY}
PROF_COLORS = {"Very well or well": TEAL, "Not well or Not at all": CORAL}
SEQ = ["#e6f4f1", "#a8dcd2", "#5cbfb0", "#0f9d8f", "#0b6e6e", "#0b2545"]   # sequential scale

REGIONS = sorted(r for r in MIG["region"].unique() if r != "Not specified")
COUNTRIES = sorted(c for c in MIG.loc[MIG["population"] > 0, "country"].unique())
REGION_COLORS = dict(zip(REGIONS, px.colors.qualitative.Safe))
NON_MAPPABLE = MIG["country"].str.contains(r", nfd|, nec|At sea", regex=True)   # not real countries

# Custom Plotly template used by every chart
pio.templates["lthc"] = go.layout.Template(pio.templates["plotly_white"])
pio.templates["lthc"].layout.update(
    font=dict(family="Inter, Segoe UI, Roboto, Arial, sans-serif", size=12.5, color="#1c2733"),
    colorway=[NAVY, TEAL, CORAL, AMBER, "#3d6fb6", "#7b5ea7", "#5f6b7a"],
    margin=dict(l=10, r=10, t=30, b=10),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, title_text=""),
)
px.defaults.template = "lthc"

LABELS = {"rate": "Prevalence (%)", "asr": "Age-standardised prevalence (%)",
          "crude_rate": "Crude prevalence (%)", "cases": "People with condition",
          "population": "Population", "age": "Age group", "sex": "Sex",
          "years": "Years in Australia", "proficiency": "Spoken English",
          "region": "Region of birth", "subregion": "Sub-region", "country": "Country of birth",
          "condition": "Condition", "language": "Language used at home"}


def empty_fig(msg="No data for the current selection"):
    """Blank figure with a message, used when a filter leaves no data."""
    fig = px.scatter()
    fig.update_layout(xaxis_visible=False, yaxis_visible=False, height=320,
                      annotations=[dict(text=msg, showarrow=False, font=dict(size=14, color=GREY))])
    return fig


def filt(df, sex=None, ages=None, regions=None, condition=None):
    """Apply the dashboard filters (empty = no filter)."""
    m = pd.Series(True, index=df.index)
    if sex:
        m &= df["sex"] == sex
    if ages:
        m &= df["age"].isin(ages)
    if regions:
        m &= df["region"].isin(regions)
    if condition:
        m &= df["condition"].isin([condition] if isinstance(condition, str) else condition)
    return df[m]


def pct_text(s):
    return s.map("{:.1f}%".format)

### 4.2 Overview charts and summary figures

In [ ]:
def kpi_summary(condition, sex, ages, regions):
    """Values for the six summary cards: list of (value, sub-text) pairs."""
    d = filt(MIG, sex, ages, regions, condition)
    if d["population"].sum() == 0:
        return [("–", "")] * 6
    tot = prevalence(d).iloc[0]
    asr = age_standardised(d.assign(_all=1), "_all", STD_W)["asr"].iloc[0]
    reg = age_standardised(d[d["region"] != "Not specified"], "region", STD_W)
    reg = reg[(reg["population"] >= 1000) & (reg["asr"] > 0)].sort_values("asr", ascending=False)
    conds = prevalence(filt(MIG, sex, ages, regions, SPECIFIC_CONDITIONS), "condition").sort_values(
        "rate", ascending=False)
    top = reg.iloc[0] if len(reg) else None
    return [
        (fmt_int(tot["population"]), "overseas-born, in selection"),
        (fmt_int(tot["cases"]), condition if len(condition) < 34 else condition[:32] + "…"),
        (fmt_pct(tot["rate"]), "cases ÷ population"),
        (fmt_pct(asr), "adjusted to overseas-born age mix"),
        (top["region"] if top is not None else "–",
         f"{fmt_pct(top['asr'])} age-standardised" if top is not None else ""),
        (conds.iloc[0]["condition"] if len(conds) else "–",
         f"{fmt_pct(conds.iloc[0]['rate'])} of population" if len(conds) else ""),
    ]


def fig_conditions(condition, sex, ages, regions):
    """Horizontal bar: prevalence of each condition, selected one highlighted."""
    conds = SPECIFIC_CONDITIONS + ([condition] if condition not in SPECIFIC_CONDITIONS else [])
    c = prevalence(filt(MIG, sex, ages, regions, conds), "condition")
    if c["population"].sum() == 0:
        return empty_fig()
    c = c.sort_values("rate")
    c["highlight"] = np.where(c["condition"] == condition, "Selected", "Other")
    fig = px.bar(c, x="rate", y="condition", orientation="h", color="highlight",
                 color_discrete_map={"Selected": CORAL, "Other": NAVY}, text=pct_text(c["rate"]),
                 hover_data={"cases": ":,", "population": ":,", "highlight": False, "rate": ":.2f"},
                 labels=LABELS)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(showlegend=False, yaxis_title=None, xaxis_title="Prevalence (%)",
                      yaxis={"categoryorder": "total ascending"})
    return fig


def fig_age_sex(condition, sex, ages, regions):
    """Grouped bar: prevalence by age group for males vs females (ignores the sex filter)."""
    a = prevalence(filt(MIG, None, ages, regions, condition).query("sex != 'Persons'"), ["age", "sex"])
    if a.empty:
        return empty_fig()
    fig = px.bar(a, x="age", y="rate", color="sex", barmode="group", color_discrete_map=SEX_COLORS,
                 category_orders={"age": AGE_ORDER, "sex": ["Male", "Female"]}, text=pct_text(a["rate"]),
                 hover_data={"cases": ":,", "population": ":,", "rate": ":.2f"}, labels=LABELS)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(yaxis_title="Prevalence (%)", xaxis_title=None)
    return fig


def fig_treemap(condition, sex, ages, regions):
    """Treemap region → sub-region → country: size = cases, colour = prevalence."""
    t = prevalence(filt(MIG, sex, ages, regions, condition), ["region", "subregion", "country"])
    t = t[t["cases"] > 0]
    if t.empty:
        return empty_fig()
    fig = px.treemap(t, path=[px.Constant("All overseas-born"), "region", "subregion", "country"],
                     values="cases", color="rate", color_continuous_scale=SEQ, labels=LABELS)
    fig.update_traces(root_color="#eef3f8",
                      hovertemplate="<b>%{label}</b><br>People with condition: %{value:,}"
                                    "<br>Prevalence: %{color:.1f}%<extra></extra>")
    fig.update_layout(margin=dict(t=10, l=0, r=0, b=0), coloraxis_colorbar_title="%")
    return fig

### 4.3 Country of birth charts

In [ ]:
def fig_world_map(condition, sex, ages, regions, min_pop=1000):
    """Choropleth: age-standardised prevalence by country of birth."""
    d = filt(MIG, sex, ages, regions, condition)
    m = age_standardised(d[~NON_MAPPABLE.loc[d.index]], "map_country", STD_W)
    m = m[m["population"] >= (min_pop or 0)]
    if m.empty:
        return empty_fig()
    fig = px.choropleth(m, locations="map_country", locationmode="country names", color="asr",
                        hover_name="map_country", color_continuous_scale=SEQ, projection="natural earth",
                        hover_data={"map_country": False, "asr": ":.1f", "crude_rate": ":.1f",
                                    "cases": ":,", "population": ":,"}, labels=LABELS)
    fig.update_geos(showcountries=True, countrycolor="#ffffff", showcoastlines=False, showframe=False,
                    landcolor="#e9edf2", bgcolor="rgba(0,0,0,0)")
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), coloraxis_colorbar=dict(title="ASR %", thickness=12))
    return fig


def fig_top_countries(condition, sex, ages, regions, min_pop=1000):
    """Horizontal bar: 15 countries with the highest age-standardised prevalence."""
    d = filt(MIG, sex, ages, regions, condition)
    c = age_standardised(d, ["country", "region"], STD_W)
    c = c[c["population"] >= (min_pop or 0)].nlargest(15, "asr").sort_values("asr")
    if c.empty:
        return empty_fig()
    fig = px.bar(c, x="asr", y="country", color="region", orientation="h", color_discrete_map=REGION_COLORS,
                 text=pct_text(c["asr"]),
                 hover_data={"crude_rate": ":.1f", "cases": ":,", "population": ":,", "asr": ":.2f"},
                 labels=LABELS)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(yaxis_title=None, xaxis_title="Age-standardised prevalence (%)",
                      yaxis={"categoryorder": "total ascending"},
                      legend=dict(orientation="h", y=-0.25, yanchor="top", x=0))
    return fig


def fig_region_compare(condition, sex, ages, regions):
    """Grouped bar: age-standardised vs crude prevalence for each region."""
    d = filt(MIG, sex, ages, regions, condition)
    r = age_standardised(d[d["region"] != "Not specified"], "region", STD_W).sort_values("asr")
    if r.empty:
        return empty_fig()
    long = r.melt(id_vars=["region", "cases", "population"], value_vars=["asr", "crude_rate"],
                  var_name="measure", value_name="value")
    long["measure"] = long["measure"].map({"asr": "Age-standardised", "crude_rate": "Crude"})
    fig = px.bar(long, x="value", y="region", color="measure", barmode="group", orientation="h",
                 color_discrete_map={"Age-standardised": TEAL, "Crude": "#c3ccd8"},
                 category_orders={"region": r["region"].tolist()[::-1],
                                  "measure": ["Age-standardised", "Crude"]},
                 hover_data={"cases": ":,", "population": ":,", "value": ":.2f"},
                 labels={**LABELS, "value": "Prevalence (%)"})
    fig.update_layout(yaxis_title=None)
    return fig


def elevated_groups(condition, sex, ages, regions):
    """Table of country × age × years groups with statistically elevated prevalence."""
    d = filt(MIG, sex, ages, None, condition)       # baseline uses all regions
    if d.empty:
        return pd.DataFrame()
    r = risk_table(d, ["country", "region", "age", "years"], ["age"], min_population=1000)
    if regions:
        r = r[r["region"].isin(regions)]
    r = r[r["risk_level"] == "Elevated"].head(100)
    return pd.DataFrame({
        "Country of birth": r["country"], "Region": r["region"], "Age group": r["age"],
        "Years in Australia": r["years"], "Population": r["population"], "Cases": r["cases"],
        "Prevalence (%)": r["rate"].round(1), "Baseline (%)": r["baseline_rate"].round(1),
        "Rate ratio": r["rate_ratio"].round(2), "z-score": r["z_score"].round(1),
        "Risk level": r["risk_level"]})

### 4.4 Years in Australia charts
Recent arrivals are younger and often healthier (the "healthy migrant effect"). For that reason, length of stay is compared **within age groups** or with age-standardised rates.

In [ ]:
def ratio_heatmap(df, split_col, numerator, denominator, ages):
    """Heatmap of rate ratios (numerator ÷ denominator group) for each condition and age group."""
    p = prevalence(df, ["condition", "age", split_col])
    w = p.pivot_table(index=["condition", "age"], columns=split_col, values="rate").reset_index()
    if numerator not in w or denominator not in w:
        return empty_fig()
    w["ratio"] = w[numerator] / w[denominator].replace(0, np.nan)
    mat = w.pivot(index="condition", columns="age", values="ratio")
    mat = mat.reindex(index=[c for c in ALL_CONDITIONS if c in mat.index],
                      columns=[a for a in AGE_ORDER if a in (ages or AGE_ORDER) and a in mat.columns])
    if mat.empty:
        return empty_fig()
    mat = mat.astype(float)
    logm = np.log2(mat).replace([np.inf, -np.inf], np.nan)      # log scale: 2× and ½× are symmetric
    finite = logm.values[np.isfinite(logm.values)]
    top = min(max(np.abs(finite).max() if finite.size else 1.0, 0.5), 3)
    fig = px.imshow(logm, aspect="auto", color_continuous_scale=[[0, TEAL], [0.5, "#f7f7f7"], [1, CORAL]],
                    zmin=-top, zmax=top, labels=dict(x="Age group", y="", color="log₂ ratio"))
    fig.update_traces(text=mat.map(lambda v: f"{v:.2f}×" if pd.notna(v) else "").values,
                      texttemplate="%{text}", customdata=mat.values,
                      hovertemplate="%{y}<br>Age %{x}<br>Rate ratio: %{customdata:.2f}<extra></extra>")
    fig.update_layout(coloraxis_colorbar=dict(title="ratio", tickvals=[-2, -1, 0, 1, 2],
                                              ticktext=["0.25×", "0.5×", "1×", "2×", "4×"], thickness=12))
    return fig


def fig_years_by_age(condition, sex, ages, regions):
    """Grouped bar: recent vs long-term residents within each age group."""
    a = prevalence(filt(MIG, sex, ages, regions, condition), ["age", "years"])
    if a["population"].sum() == 0:
        return empty_fig()
    fig = px.bar(a, x="age", y="rate", color="years", barmode="group", color_discrete_map=YEARS_COLORS,
                 category_orders={"age": AGE_ORDER, "years": YEARS_ORDER}, text=pct_text(a["rate"]),
                 hover_data={"cases": ":,", "population": ":,", "rate": ":.2f"}, labels=LABELS)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(yaxis_title="Prevalence (%)", xaxis_title=None)
    return fig


def fig_years_by_region(condition, sex, ages, regions):
    """Slope chart: age-standardised prevalence for each region, recent vs long-term."""
    d = filt(MIG, sex, ages, regions, condition)
    r = age_standardised(d[d["region"] != "Not specified"], ["region", "years"], STD_W)
    if r.empty:
        return empty_fig()
    fig = px.line(r, x="years", y="asr", color="region", markers=True, color_discrete_map=REGION_COLORS,
                  category_orders={"years": YEARS_ORDER},
                  hover_data={"cases": ":,", "population": ":,", "asr": ":.2f"}, labels=LABELS)
    fig.update_traces(line_width=2.5, marker_size=9)
    fig.update_layout(xaxis_title=None, yaxis_title="Age-standardised prevalence (%)",
                      legend=dict(orientation="v", x=1.02, y=1, yanchor="top"))
    return fig


def fig_years_heatmap(condition, sex, ages, regions):
    return ratio_heatmap(filt(MIG, sex, ages, regions), "years", "More than 10 years", "0–10 years", ages)

### 4.5 Language and English proficiency charts

In [ ]:
def fig_prof_by_age(condition, sex, ages, regions):
    """Grouped bar: English very well vs not well, within each age group."""
    a = prevalence(filt(LANG, sex, ages, regions, condition), ["age", "proficiency"])
    if a["population"].sum() == 0:
        return empty_fig()
    fig = px.bar(a, x="age", y="rate", color="proficiency", barmode="group", color_discrete_map=PROF_COLORS,
                 category_orders={"age": AGE_ORDER, "proficiency": PROF_ORDER}, text=pct_text(a["rate"]),
                 hover_data={"cases": ":,", "population": ":,", "rate": ":.2f"}, labels=LABELS)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(yaxis_title="Prevalence (%)", xaxis_title=None)
    return fig


def fig_prof_heatmap(condition, sex, ages, regions):
    return ratio_heatmap(filt(LANG, sex, ages, regions), "proficiency",
                         "Not well or Not at all", "Very well or well", ages)


def fig_language_dumbbell(condition, sex, ages, regions, min_pop=1000):
    """Dumbbell: prevalence by language, one dot per English level (20 largest languages)."""
    d = filt(LANG, sex, ages, regions, condition)
    l = age_standardised(d, ["language", "proficiency"], STD_W)
    l = l[l["population"] >= (min_pop or 0)]
    l = l[l["language"].isin(l.groupby("language")["population"].sum().nlargest(20).index)]
    if l.empty:
        return empty_fig("No language groups meet the minimum population")
    order = l.groupby("language")["asr"].max().sort_values().index.tolist()
    fig = px.line(l.sort_values(["language", "asr"]), x="asr", y="language", line_group="language",
                  color_discrete_sequence=["#c3ccd8"])
    fig.update_traces(hoverinfo="skip", hovertemplate=None, line_width=3)
    dots = px.scatter(l, x="asr", y="language", color="proficiency", color_discrete_map=PROF_COLORS,
                      size="population", size_max=18, category_orders={"proficiency": PROF_ORDER},
                      hover_data={"crude_rate": ":.1f", "cases": ":,", "population": ":,", "asr": ":.2f"},
                      labels=LABELS)
    for tr in dots.data:
        fig.add_trace(tr)
    fig.update_layout(xaxis_title="Age-standardised prevalence (%)", yaxis_title=None,
                      yaxis=dict(categoryorder="array", categoryarray=order))
    return fig


def explorer_table(condition, sex, ages, regions, view="migration"):
    """Filtered rows for the Data explorer tab."""
    src = MIG if view == "migration" else LANG
    d = filt(src, sex, ages, regions, condition)
    d = d[d["population"] > 0]
    keep = ["country", "region", "subregion", "age", "sex", "condition", "cases", "population", "rate"]
    keep.insert(3, "years" if view == "migration" else "proficiency")
    if view == "language":
        keep.insert(3, "language")
    return d[keep].assign(rate=lambda x: x["rate"].round(2)).sort_values(["country", "age"])

### 4.6 Preview the charts in the notebook
Change the filters below and re-run to explore. These are the same figures the dashboard shows.

In [ ]:
F = dict(condition=ANY_LTHC, sex="Persons", ages=AGE_ORDER, regions=[])   # preview filters

for label, (value, sub) in zip(["Population", "People with condition", "Crude prevalence",
                                "Age-standardised", "Highest-risk region", "Most common condition"],
                               kpi_summary(**F)):
    print(f"{label:<24} {value:<28} {sub}")

In [ ]:
fig_conditions(**F).show()
fig_age_sex(**F).show()

In [ ]:
fig_treemap(**F).show()

In [ ]:
fig_world_map(**F).show()
fig_top_countries(**F).show()
fig_region_compare(**F).show()

In [ ]:
elevated_groups(**F).head(10)

In [ ]:
fig_years_by_age(**F).show()
fig_years_by_region(**F).show()
fig_years_heatmap(**F).show()

In [ ]:
fig_prof_by_age(**F).show()
fig_prof_heatmap(**F).show()
fig_language_dumbbell(**F).show()

## 5. Analysis dashboard

The dashboard is a **Dash** app that arranges the Plotly Express charts above into five tabs with global filters. It is shown **inside the notebook** (`jupyter_mode="inline"`). Change to `"external"` to open it in a browser at http://127.0.0.1:8050 instead.

### 5.1 Styling (CSS)
The CSS is kept in the notebook and injected into both apps, so no separate stylesheet file is needed.

In [ ]:
CSS = """
/* ---------- LTHC dashboard theme ---------- */
:root {
  --navy: #0b2545;
  --navy-2: #13315c;
  --teal: #0f9d8f;
  --coral: #e4572e;
  --amber: #f2a541;
  --bg: #f4f6fa;
  --card: #ffffff;
  --text: #1c2733;
  --muted: #5f6b7a;
  --border: #e3e8ef;
  --radius: 12px;
  --shadow: 0 1px 2px rgba(16, 24, 40, .05), 0 1px 3px rgba(16, 24, 40, .08);
}

* { box-sizing: border-box; }

body {
  margin: 0;
  background: var(--bg);
  color: var(--text);
  font-family: "Inter", "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  font-size: 14px;
}

/* Header */
.header {
  background: linear-gradient(120deg, var(--navy) 0%, var(--navy-2) 60%, #1d5c78 100%);
  color: #fff;
  padding: 22px 32px 18px;
}
.header h1 { margin: 0; font-size: 24px; font-weight: 700; letter-spacing: -.2px; }
.header p  { margin: 6px 0 0; color: #c9d6e8; font-size: 13.5px; }
.header .pill {
  display: inline-block; margin-right: 8px; margin-top: 10px;
  padding: 3px 10px; border-radius: 999px; font-size: 12px;
  background: rgba(255, 255, 255, .12); color: #e6eef8;
}

/* Filter bar */
.filter-bar {
  position: sticky; top: 0; z-index: 50;
  display: grid; grid-template-columns: 2.2fr 1.2fr 1.6fr 2.4fr; gap: 16px;
  background: var(--card); border-bottom: 1px solid var(--border);
  padding: 12px 32px; box-shadow: var(--shadow);
}
.filter label.title, .control-label {
  display: block; font-size: 11px; font-weight: 700; text-transform: uppercase;
  letter-spacing: .6px; color: var(--muted); margin-bottom: 6px;
}
.radio-inline label, .check-inline label { margin-right: 14px; font-size: 13.5px; cursor: pointer; }
.radio-inline input, .check-inline input { margin-right: 5px; accent-color: var(--teal); }

/* Page */
.page { padding: 18px 32px 32px; max-width: 1600px; margin: 0 auto; }

/* KPI cards */
.kpi-row { display: grid; grid-template-columns: repeat(6, 1fr); gap: 14px; margin-bottom: 16px; }
.kpi {
  background: var(--card); border: 1px solid var(--border); border-radius: var(--radius);
  padding: 14px 16px; box-shadow: var(--shadow); border-top: 3px solid var(--teal);
}
.kpi .label { font-size: 11.5px; color: var(--muted); font-weight: 600; text-transform: uppercase; letter-spacing: .4px; }
.kpi .value { font-size: 24px; font-weight: 700; margin-top: 4px; color: var(--navy); }
.kpi .sub   { font-size: 12px; color: var(--muted); margin-top: 2px; }
.kpi.coral { border-top-color: var(--coral); }
.kpi.amber { border-top-color: var(--amber); }
.kpi.navy  { border-top-color: var(--navy); }

/* Tabs */
.tabs-container { margin-bottom: 14px; }
.tab {
  border: none !important; background: transparent !important;
  padding: 10px 16px !important; color: var(--muted) !important; font-weight: 600;
  border-bottom: 3px solid transparent !important;
}
.tab--selected {
  color: var(--navy) !important; border-bottom: 3px solid var(--teal) !important;
  background: transparent !important;
}

/* Cards & grid */
.grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
.grid-3-2 { display: grid; grid-template-columns: 3fr 2fr; gap: 16px; }
.grid-1-2 { display: grid; grid-template-columns: 1fr 2fr; gap: 16px; }
.card {
  background: var(--card); border: 1px solid var(--border); border-radius: var(--radius);
  padding: 14px 16px 6px; box-shadow: var(--shadow); margin-bottom: 16px; min-width: 0;
}
.card h3 { margin: 0 0 2px; font-size: 15px; color: var(--navy); }
.card .caption { margin: 0 0 6px; font-size: 12.5px; color: var(--muted); }
.card-controls { display: flex; gap: 24px; align-items: center; margin: 6px 0 8px; flex-wrap: wrap; }
.card-controls .slider { min-width: 320px; flex: 1; }

/* Risk identifier */
.risk-inputs { display: grid; grid-template-columns: 2fr 1.3fr 1.3fr 1fr 1.6fr; gap: 16px; }
.risk-summary { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin: 14px 0 4px; }
.badge {
  display: inline-block; padding: 4px 12px; border-radius: 999px;
  font-weight: 700; font-size: 13px; color: #fff;
}
.badge.Elevated { background: var(--coral); }
.badge.Typical  { background: #8a96a8; }
.badge.Lower    { background: var(--teal); }
.badge.Insufficient { background: #b9c1cc; }

/* Notes */
.note {
  font-size: 12.5px; color: var(--muted); background: #eef3f8; border-left: 3px solid var(--navy-2);
  padding: 10px 14px; border-radius: 6px; margin: 4px 0 14px;
}
.footer { font-size: 12px; color: var(--muted); text-align: center; padding: 10px 0 24px; }

/* Responsive */
@media (max-width: 1200px) {
  .kpi-row { grid-template-columns: repeat(3, 1fr); }
  .filter-bar, .risk-inputs { grid-template-columns: 1fr 1fr; }
}
@media (max-width: 900px) {
  .grid-2, .grid-3-2, .grid-1-2, .risk-summary { grid-template-columns: 1fr; }
  .kpi-row { grid-template-columns: repeat(2, 1fr); }
  .filter-bar, .risk-inputs { grid-template-columns: 1fr; position: static; }
  .page, .header, .filter-bar { padding-left: 16px; padding-right: 16px; }
}

/* ---------- Personal risk check ---------- */
.profile-card h3 { font-size: 18px; }
.profile-row { display: grid; gap: 18px; margin-top: 12px; }
.profile-row.three { grid-template-columns: 1fr 1.3fr 1.3fr; }
.profile-row.four  { grid-template-columns: 1.3fr 1.2fr 1.6fr 1.9fr; margin-bottom: 10px; }

.headline { display: grid; grid-template-columns: 2.2fr 1fr; gap: 24px; align-items: center; padding: 18px 22px; }
.headline .label { font-size: 11.5px; color: var(--muted); font-weight: 700; text-transform: uppercase; letter-spacing: .5px; }
.profile-text { font-size: 19px; font-weight: 600; color: var(--navy); margin: 6px 0 10px; line-height: 1.35; }
.headline-right { border-left: 1px solid var(--border); padding-left: 24px; }
.headline .big { font-size: 46px; font-weight: 800; color: var(--coral); line-height: 1.1; margin-top: 4px; }
.headline .sub { font-size: 13px; color: var(--muted); }
.badge-row { display: flex; flex-wrap: wrap; align-items: center; gap: 10px; margin-top: 8px; }
.badge-row .rel { font-size: 12.5px; color: var(--muted); }
.warn { width: 100%; font-size: 12.5px; color: #8a5a00; background: #fff6e0; padding: 6px 10px; border-radius: 6px; }

.section-title { font-size: 13px; font-weight: 700; color: var(--navy); text-transform: uppercase; letter-spacing: .5px; margin: 4px 0 10px; }
.top-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 14px; margin-bottom: 16px; }
.cond-card .label { font-size: 13px; text-transform: none; color: var(--text); letter-spacing: 0; }
.cond-card .value { font-size: 30px; }
.cond-card.Elevated { border-top-color: var(--coral); }
.cond-card.Typical  { border-top-color: #8a96a8; }
.cond-card.Lower    { border-top-color: var(--teal); }

@media (max-width: 1200px) {
  .profile-row.three, .profile-row.four { grid-template-columns: 1fr 1fr; }
}
@media (max-width: 900px) {
  .profile-row.three, .profile-row.four, .headline, .top-grid { grid-template-columns: 1fr; }
  .headline-right { border-left: none; padding-left: 0; }
}

/* ---------- Header link (both apps) ---------- */
.header-flex { display: flex; justify-content: space-between; align-items: flex-start; gap: 24px; }
.header-link {
  flex-shrink: 0; margin-top: 4px; padding: 9px 16px; border-radius: 8px;
  background: rgba(255, 255, 255, .14); color: #fff; text-decoration: none; font-weight: 600; font-size: 13.5px;
  border: 1px solid rgba(255, 255, 255, .25);
}
.header-link:hover { background: rgba(255, 255, 255, .24); }

/* ---------- Risk Identifier app: form layout ---------- */
.form-layout { display: grid; grid-template-columns: 380px minmax(0, 1fr); gap: 20px; align-items: start; }
.form-card { position: sticky; top: 16px; padding: 18px 20px 20px; }
.form-card h3 { font-size: 17px; }
.step { border-top: 1px solid var(--border); padding: 14px 0 4px; }
.step-title { display: flex; align-items: center; gap: 10px; font-weight: 700; color: var(--navy); margin-bottom: 10px; }
.step-num {
  display: inline-flex; align-items: center; justify-content: center; width: 24px; height: 24px;
  border-radius: 50%; background: var(--teal); color: #fff; font-size: 12.5px; font-weight: 700;
}
.form-field { margin-bottom: 14px; }
.hint { font-size: 12px; color: var(--muted); margin-top: 4px; }
.radio-stack label { display: block; padding: 3px 0; font-size: 14px; cursor: pointer; }
.radio-stack input { margin-right: 8px; accent-color: var(--teal); }
.form-error { color: var(--coral); font-weight: 600; font-size: 13px; min-height: 18px; margin: 4px 0 8px; }
.btn-primary {
  width: 100%; padding: 12px 16px; border: none; border-radius: 10px; cursor: pointer;
  background: var(--coral); color: #fff; font-size: 15px; font-weight: 700; letter-spacing: .2px;
  box-shadow: 0 2px 6px rgba(228, 87, 46, .35);
}
.btn-primary:hover { filter: brightness(1.06); }

.placeholder { text-align: center; padding: 70px 30px; color: var(--muted); }
.placeholder h3 { font-size: 18px; margin-top: 10px; }
.placeholder-icon {
  width: 64px; height: 64px; margin: 0 auto; border-radius: 50%; background: #e6f4f1; color: var(--teal);
  display: flex; align-items: center; justify-content: center; font-size: 30px; font-weight: 800;
}
.cond-card { position: relative; }
.cond-card .rank { position: absolute; top: 12px; right: 14px; font-weight: 800; color: #c3ccd8; font-size: 18px; }

@media (max-width: 1100px) {
  .form-layout { grid-template-columns: 1fr; }
  .form-card { position: static; }
}
@media (max-width: 700px) {
  .header-flex { flex-direction: column; }
}
"""


def page_template(css):
    """Dash page template with the CSS embedded."""
    return ("<!DOCTYPE html><html><head>{%metas%}<title>{%title%}</title>{%favicon%}{%css%}"
            "<style>" + css + "</style></head><body>{%app_entry%}"
            "<footer>{%config%}{%scripts%}{%renderer%}</footer></body></html>")

### 5.2 Layout building blocks

In [ ]:
GRAPH_CFG = {"displaylogo": False, "responsive": True,
             "modeBarButtonsToRemove": ["lasso2d", "select2d"],
             "toImageButtonOptions": {"format": "png", "scale": 2}}   # camera icon saves a PNG
TABLE_STYLE = dict(
    style_table={"overflowX": "auto"},
    style_cell={"fontFamily": "Inter, Segoe UI, Arial", "fontSize": 13, "padding": "6px 10px",
                "textAlign": "left", "border": "none", "borderBottom": "1px solid #e3e8ef"},
    style_header={"fontWeight": 700, "backgroundColor": "#f4f6fa", "color": NAVY},
)


def card(title, caption, *children, controls=None):
    """White card with a title, caption, optional controls and content."""
    body = [html.H3(title), html.P(caption, className="caption")]
    if controls is not None:
        body.append(html.Div(controls, className="card-controls"))
    return html.Div(body + list(children), className="card")


def graph(id_, height=380):
    return dcc.Graph(id=id_, config=GRAPH_CFG, style={"height": f"{height}px"})


def kpi(id_, label, cls=""):
    """Summary tile; its value and sub-text are filled in by a callback."""
    return html.Div([html.Div(label, className="label"),
                     html.Div("–", id=f"{id_}-value", className="value"),
                     html.Div("", id=f"{id_}-sub", className="sub")], className=f"kpi {cls}")


def min_pop_slider(id_, value=1000):
    marks = {0: "0", 500: "500", 1000: "1k", 5000: "5k", 10000: "10k", 20000: "20k"}
    return html.Div([html.Label("Minimum population (reliability filter)", className="control-label"),
                     dcc.Slider(id=id_, min=0, max=20000, step=None, marks=marks, value=value)],
                    className="slider")


def tab(label, value, children):
    return dcc.Tab(label=label, value=value, className="tab", selected_className="tab--selected",
                   children=children)

### 5.3 Dashboard layout: header, filters, summary tiles and five tabs

In [ ]:
dashboard_app = Dash("lthc_dashboard", title="LTHC Analysis Dashboard", suppress_callback_exceptions=True)
dashboard_app.index_string = page_template(CSS)

header = html.Div([
    html.Div([
        html.H1("Long-Term Health Condition Analysis Dashboard"),
        html.P("Prevalence of long-term health conditions among overseas-born Australians by country of "
               "birth, years in Australia, language used at home and English proficiency."),
        html.Span("Census 2021", className="pill"),
        html.Span("AIHW Tables 10, 13 & 14", className="pill"),
        html.Span(f"{len(COUNTRIES)} countries of birth", className="pill"),
        html.Span(f"{LANG['language'].nunique()} languages", className="pill"),
    ]),
    html.A("Open Risk Identifier →", href="http://127.0.0.1:8052", target="_blank", className="header-link"),
], className="header header-flex")

filter_bar = html.Div([
    html.Div([html.Label("Health condition", className="title"),
              dcc.Dropdown(id="f-condition", options=ALL_CONDITIONS, value=ANY_LTHC, clearable=False)]),
    html.Div([html.Label("Sex", className="title"),
              dcc.RadioItems(id="f-sex", options=["Persons", "Male", "Female"], value="Persons",
                             inline=True, className="radio-inline")]),
    html.Div([html.Label("Age group", className="title"),
              dcc.Checklist(id="f-age", options=AGE_ORDER, value=AGE_ORDER, inline=True,
                            className="check-inline")]),
    html.Div([html.Label("Region of birth", className="title"),
              dcc.Dropdown(id="f-region", options=REGIONS, value=[], multi=True, placeholder="All regions")]),
], className="filter-bar")

KPI_IDS = ["k-pop", "k-cases", "k-crude", "k-asr", "k-region", "k-top"]
kpis = html.Div([
    kpi("k-pop", "Population in scope", "navy"), kpi("k-cases", "People with condition"),
    kpi("k-crude", "Crude prevalence"), kpi("k-asr", "Age-standardised prevalence", "amber"),
    kpi("k-region", "Highest-risk region", "coral"), kpi("k-top", "Most common condition"),
], className="kpi-row")

tabs = dcc.Tabs(id="tabs", value="overview", className="tabs-container", children=[
    tab("Overview", "overview", [
        html.Div([
            card("Prevalence by condition", "Share of the selected population reporting each condition. "
                 "Selected condition highlighted.", graph("g-conditions", 420)),
            card("Age and sex profile", "Age-specific prevalence of the selected condition for males and "
                 "females.", graph("g-age-sex", 420)),
        ], className="grid-2"),
        card("Where the burden sits", "Region → sub-region → country of birth. Box size = people with the "
             "condition; colour = prevalence. Click to drill down.", graph("g-treemap", 520)),
    ]),
    tab("Country of birth", "geo", [
        card("Global view", "Age-standardised prevalence by country of birth (UK countries combined).",
             graph("g-map", 520), controls=[min_pop_slider("s-geo-minpop")]),
        html.Div([
            card("Highest-prevalence countries", "Top 15 countries by age-standardised prevalence.",
                 graph("g-top-countries", 480)),
            card("Region comparison", "Age-standardised vs crude prevalence. The gap shows how much age "
                 "structure alone drives differences.", graph("g-regions", 480)),
        ], className="grid-2"),
        card("Population groups with elevated risk",
             "Country × age × length-of-stay groups ranked by z-score against the same age group. "
             "Elevated = z ≥ 3 and rate ratio ≥ 1.2. Only groups with ≥ 1,000 people are assessed.",
             dash_table.DataTable(id="t-risk", page_size=12, sort_action="native", filter_action="native",
                                  **TABLE_STYLE)),
    ]),
    tab("Years in Australia", "migration", [
        html.Div("Recent arrivals are typically younger and healthier (the ‘healthy migrant effect’), so "
                 "comparisons are made within age groups or age-standardised.", className="note"),
        html.Div([
            card("Length of stay by age group", "Prevalence for recent (0–10 years) vs long-term (>10 years) "
                 "residents.", graph("g-years-age", 400)),
            card("Length of stay by region", "Age-standardised prevalence; each line joins the same region.",
                 graph("g-years-region", 400)),
        ], className="grid-2"),
        card("Which conditions increase with length of stay?", "Rate ratio = prevalence (>10 years) ÷ "
             "prevalence (0–10 years), within each age group.", graph("g-years-heat", 460)),
    ]),
    tab("Language & English", "language", [
        html.Div([
            card("English proficiency by age group", "Prevalence among people who speak English very well/"
                 "well vs not well/not at all.", graph("g-prof-age", 400)),
            card("Proficiency gap by condition", "Rate ratio = prevalence (not well) ÷ prevalence (very "
                 "well), within age groups.", graph("g-prof-heat", 400)),
        ], className="grid-2"),
        card("Language used at home", "Age-standardised prevalence by language and English proficiency "
             "(20 largest language groups).", graph("g-lang-dumbbell", 560),
             controls=[min_pop_slider("s-lang-minpop")]),
    ]),
    tab("Data explorer", "data", [
        card("Underlying data", "Filtered by the controls above. Use the column filters and export to CSV.",
             dash_table.DataTable(id="t-data", page_size=15, sort_action="native", filter_action="native",
                                  export_format="csv", export_headers="display", **TABLE_STYLE),
             controls=[dcc.RadioItems(id="d-view", value="migration", className="radio-inline", options=[
                 {"label": "Years in Australia (Table 10)", "value": "migration"},
                 {"label": "Language & English proficiency (Tables 13/14)", "value": "language"}])]),
    ]),
])

dashboard_app.layout = html.Div([
    header, filter_bar,
    html.Div([
        kpis, tabs,
        html.Div("Ethics & data notes: aggregated census counts only – no identifiable data. Small cells are "
                 "randomly perturbed by the ABS, so Male + Female may not equal Persons. Results describe "
                 "associations, not causes, and must not be used to make decisions about individuals.",
                 className="note"),
        html.Div("Source: Australian Institute of Health and Welfare, Census 2021.", className="footer"),
    ], className="page"),
])

### 5.4 Callbacks
Each callback runs when a filter changes and calls the chart functions from Section 4.

In [ ]:
FILTERS = [Input("f-condition", "value"), Input("f-sex", "value"),
           Input("f-age", "value"), Input("f-region", "value")]


def table_columns(df):
    return [{"name": c, "id": c, "type": "numeric"} if df[c].dtype.kind in "fi" else {"name": c, "id": c}
            for c in df.columns]


@dashboard_app.callback([Output(f"{k}-{p}", "children") for k in KPI_IDS for p in ("value", "sub")], FILTERS)
def update_kpis(condition, sex, ages, regions):
    return [x for pair in kpi_summary(condition, sex, ages, regions) for x in pair]


@dashboard_app.callback(Output("g-conditions", "figure"), Output("g-age-sex", "figure"),
                        Output("g-treemap", "figure"), FILTERS)
def update_overview(*f):
    return fig_conditions(*f), fig_age_sex(*f), fig_treemap(*f)


@dashboard_app.callback(Output("g-map", "figure"), Output("g-top-countries", "figure"),
                        Output("g-regions", "figure"), FILTERS + [Input("s-geo-minpop", "value")])
def update_geo(condition, sex, ages, regions, min_pop):
    f = (condition, sex, ages, regions)
    return fig_world_map(*f, min_pop), fig_top_countries(*f, min_pop), fig_region_compare(*f)


@dashboard_app.callback(Output("t-risk", "data"), Output("t-risk", "columns"), FILTERS)
def update_risk_table(*f):
    t = elevated_groups(*f)
    return t.to_dict("records"), table_columns(t)


@dashboard_app.callback(Output("g-years-age", "figure"), Output("g-years-region", "figure"),
                        Output("g-years-heat", "figure"), FILTERS)
def update_migration(*f):
    return fig_years_by_age(*f), fig_years_by_region(*f), fig_years_heatmap(*f)


@dashboard_app.callback(Output("g-prof-age", "figure"), Output("g-prof-heat", "figure"),
                        Output("g-lang-dumbbell", "figure"), FILTERS + [Input("s-lang-minpop", "value")])
def update_language(condition, sex, ages, regions, min_pop):
    f = (condition, sex, ages, regions)
    return fig_prof_by_age(*f), fig_prof_heatmap(*f), fig_language_dumbbell(*f, min_pop)


@dashboard_app.callback(Output("t-data", "data"), Output("t-data", "columns"), FILTERS + [Input("d-view", "value")])
def update_table(condition, sex, ages, regions, view):
    t = explorer_table(condition, sex, ages, regions, view)
    cols = [{"name": LABELS.get(c, c), "id": c, "type": "numeric"} if c in ("cases", "population", "rate")
            else {"name": LABELS.get(c, c), "id": c} for c in t.columns]
    return t.to_dict("records"), cols

### 5.5 Show the dashboard

In [ ]:
dashboard_app.run(jupyter_mode="inline", jupyter_height=1400, port=8050, debug=False)

## 6. Risk Identifier

The Risk Identifier is a separate form-based app. The user describes a person (age, sex, birthplace, years in Australia, language, English level) and sees which long-term conditions people with that background are most likely to report.

**How the estimate works:**

1. It takes the census rate for the person's age, sex and years in Australia. It then refines that rate from region → sub-region → country with **empirical-Bayes smoothing**, so small groups borrow strength from their wider region.
2. If a language or English level is chosen, it multiplies the result by that group's relative rate, taken from the language table.

The census doesn't cross years in Australia with language, so the two effects are assumed to be independent.

### 6.1 The estimator

In [ ]:
ANY = "Any"
SMOOTHING_K = 200   # pseudo-population used to shrink small groups towards a broader group


def _rates(df: pd.DataFrame, conds: list[str]) -> pd.DataFrame:
    """cases, population and rate per condition (all conditions returned, zeros if absent)."""
    g = df.groupby("condition")[["cases", "population"]].sum().reindex(conds, fill_value=0).astype(float)
    g["rate"] = np.where(g["population"] > 0, g["cases"] / g["population"], np.nan)
    return g


def _smooth(group: pd.DataFrame, prior_rate: pd.Series, k: float = SMOOTHING_K) -> pd.Series:
    """Empirical-Bayes shrinkage: (cases + k*prior) / (population + k)."""
    prior = prior_rate.fillna(0)
    return (group["cases"] + k * prior) / (group["population"] + k)


def estimate_profile(mig: pd.DataFrame, lang: pd.DataFrame, *, age: str, sex: str,
                     region: str = ANY, country: str = ANY, years: str = ANY,
                     language: str = ANY, proficiency: str = ANY,
                     conditions: list[str] | None = None) -> tuple[pd.DataFrame, dict]:
    """
    Estimate how likely a person with the given profile is to report each
    long-term health condition.

    Step 1 – place of birth & length of stay (Table 10 view)
        Hierarchical empirical-Bayes estimate. Start from all overseas-born
        people of the same age, sex (and years in Australia), then move down
        region -> sub-region -> country. At each level
            estimate = (cases + k * previous_estimate) / (population + k),  k = 200
        so large groups speak for themselves and small groups borrow strength
        from the wider area.
        Where a sex-specific cell was not published, the both-sexes (Persons)
        figure is used, scaled by the overall male/female difference for that
        age group and condition.
    Step 2 – language & English proficiency (Tables 13/14 view)
        Relative effect = smoothed prevalence for the chosen language / English
        level ÷ prevalence for everyone of the same birthplace, age and sex in
        the language table. The Step 1 estimate is multiplied by this ratio.
        The census does not cross years-in-Australia with language, so the two
        effects are combined assuming they act independently.
    Comparison
        "Average" = all overseas-born people of the same age group and sex.

    Returns (per-condition DataFrame, info dict).
    """
    conds = conditions or ([ANY_LTHC] + SPECIFIC_CONDITIONS)
    info: dict = {"notes": []}

    m_age = mig[mig["age"] == age]
    l_age = lang[lang["age"] == age]
    if years != ANY:
        m_age = m_age[m_age["years"] == years]

    # overall sex adjustment factor (used when a sex-specific cell is missing)
    def _sex_factor(df):
        if sex == "Persons":
            return pd.Series(1.0, index=conds)
        r_s = _rates(df[df["sex"] == sex], conds)["rate"]
        r_p = _rates(df[df["sex"] == "Persons"], conds)["rate"]
        return (r_s / r_p).replace([np.inf, -np.inf], np.nan).fillna(1.0)

    f_mig = _sex_factor(m_age)
    f_lang = _sex_factor(l_age)

    def rates_sex(df, factor):
        """Rates for the chosen sex, falling back to scaled Persons figures when unpublished."""
        r = _rates(df[df["sex"] == sex], conds)
        r["fallback"] = False
        if sex != "Persons":
            p = _rates(df[df["sex"] == "Persons"], conds)
            miss = (r["population"] == 0) & (p["population"] > 0)
            r.loc[miss, "cases"] = p.loc[miss, "cases"] * factor[miss]
            r.loc[miss, "population"] = p.loc[miss, "population"]
            r.loc[miss, "fallback"] = True
            r["rate"] = np.where(r["population"] > 0, r["cases"] / r["population"], np.nan)
        return r

    # ---- Step 1: place of birth and years in Australia -------------------
    avg_all = rates_sex(mig[mig["age"] == age], _sex_factor(mig[mig["age"] == age]))
    start = rates_sex(m_age, f_mig)
    est = start["rate"].fillna(avg_all["rate"]).fillna(0)

    levels = []
    if country != ANY:
        row = mig.loc[mig["country"] == country, ["region", "subregion"]].iloc[0]
        levels = [("region", row["region"]), ("subregion", row["subregion"]), ("country", country)]
        info["place"] = country
    elif region != ANY:
        levels = [("region", region)]
        info["place"] = region
    else:
        info["place"] = "any country"

    own = pd.Series(True, index=conds)
    for col, val in levels:
        r = rates_sex(m_age[m_age[col] == val], f_mig)
        est = (r["cases"] + SMOOTHING_K * est) / (r["population"] + SMOOTHING_K)
        own = r["population"] > 0

    # size of the matched census group (same for every condition)
    grp_rows = m_age[(m_age["sex"] == sex) & (m_age["condition"] == ANY_LTHC)]
    if levels:
        col, val = levels[-1]
        grp_rows = grp_rows[grp_rows[col] == val]
    info["population"] = int(grp_rows["group_population"].sum())

    # ---- Step 2: language used at home and English proficiency -----------
    ratio = pd.Series(1.0, index=conds)
    if language != ANY or proficiency != ANY:
        place_l = l_age
        if levels:
            col, val = levels[-1]
            place_l = place_l[place_l[col] == val]
        sel = place_l
        if language != ANY:
            sel = sel[sel["language"] == language]
        if proficiency != ANY:
            sel = sel[sel["proficiency"] == proficiency]
        ref = rates_sex(place_l, f_lang)
        s = rates_sex(sel, f_lang)
        info["language_population"] = int(
            sel.loc[(sel["sex"] == sex) & (sel["condition"] == ANY_LTHC), "group_population"].sum())
        if s["population"].max() > 0:
            ref_rate = ref["rate"]
            ratio = (_smooth(s, ref_rate) / ref_rate.replace(0, np.nan))
            ratio = ratio.replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(0.25, 4)
        else:
            info["notes"].append("No census records for this language / English combination – "
                                 "language effect not applied.")

    likelihood = (est * ratio).clip(upper=0.99) * 100
    average = avg_all["rate"] * 100
    out = pd.DataFrame({
        "condition": conds,
        "likelihood": likelihood.values,
        "average": average.values,
        "language_effect": ratio.values,
        "own_data": own.values if levels else True,
    })
    out["relative"] = np.where(out["average"] > 0, out["likelihood"] / out["average"], np.nan)
    out["level"] = np.select(
        [out["average"].isna() | (out["average"] <= 0), out["relative"] >= 1.2, out["relative"] <= 0.8],
        ["Not reported", "Higher than average", "Lower than average"], "About average")
    out["one_in"] = np.where(out["likelihood"] > 0, np.round(100 / out["likelihood"]), np.nan)

    n = info["population"]
    info["confidence"] = "High" if n >= 5000 else "Medium" if n >= 500 else "Low"
    if n < 500:
        info["notes"].append("Small census group – the estimate leans on the wider region.")
    if levels and not bool(own.all()):
        info["notes"].append("Some conditions were not published for this exact group; "
                             "those estimates come from the wider sub-region / region.")
    return out, info

### 6.2 Result text and charts

In [ ]:
LEVEL_COLORS = {"Higher than average": CORAL, "About average": GREY, "Lower than average": TEAL,
                "Not reported": "#c3ccd8"}
LEVEL_CLASS = {"Higher than average": "Elevated", "About average": "Typical", "Lower than average": "Lower",
               "Not reported": "Insufficient"}


def one_in(pct):
    """Turn a percentage into plain words, e.g. 19% -> 'about 1 in 5'."""
    if not pct or pd.isna(pct) or pct <= 0:
        return "–"
    if pct >= 50:
        return f"about {round(pct / 10)} in 10"
    n = 100 / pct
    return f"about 1 in {n:,.0f}" if n < 1000 else "fewer than 1 in 1,000"


def describe(age, sex, region, country, years, language, prof):
    """Plain-English description of the profile."""
    who = {"Persons": "A person", "Male": "A man", "Female": "A woman"}[sex]
    bits = [f"{who} {'aged 65 or over' if age == '65 and over' else 'aged ' + age}, " +
            (f"born in {country}" if country != ANY else f"born in {region}" if region != ANY else "born overseas")]
    if years != ANY:
        bits.append("living in Australia for " + ("up to 10 years" if years == "0–10 years" else "more than 10 years"))
    if language != ANY:
        bits.append(f"speaking {language} at home")
    if prof != ANY:
        bits.append("who speaks English " + ("very well or well" if prof.startswith("Very") else "not well or not at all"))
    return ", ".join(bits)


def fig_likelihood(spec):
    """Bar: estimated likelihood of each condition, with the average shown as a diamond."""
    plot = spec.sort_values("likelihood")
    fig = px.bar(plot, x="likelihood", y="condition", orientation="h", color="level",
                 color_discrete_map=LEVEL_COLORS, text=pct_text(plot["likelihood"]),
                 category_orders={"condition": plot["condition"].tolist()[::-1], "level": list(LEVEL_COLORS)},
                 hover_data={"average": ":.1f", "relative": ":.2f", "likelihood": ":.1f", "level": False},
                 labels={"likelihood": "Estimated likelihood (%)", "condition": "", "average": "Average (%)",
                         "relative": "Times the average", "level": ""})
    fig.update_traces(textposition="outside", cliponaxis=False)
    avg = px.scatter(plot, x="average", y="condition", labels={"average": "Average (%)", "condition": ""})
    avg.update_traces(marker=dict(symbol="diamond", size=11, color=NAVY, line=dict(color="white", width=1)),
                      name="Average (same age & sex)", showlegend=True)
    for tr in avg.data:
        fig.add_trace(tr)
    fig.update_layout(xaxis_title="Estimated % of people with this profile", yaxis_title=None)
    return fig


def fig_relative(spec):
    """Bar on a log scale: how many times more/less likely than the average."""
    rel = spec[spec["relative"].notna() & (spec["relative"] > 0)].sort_values("relative")
    if rel.empty:
        return empty_fig("Not enough data to compare")
    rel = rel.assign(log_rel=np.log2(rel["relative"]))
    fig = px.bar(rel, x="log_rel", y="condition", orientation="h", color="level", color_discrete_map=LEVEL_COLORS,
                 category_orders={"condition": rel["condition"].tolist()[::-1], "level": list(LEVEL_COLORS)},
                 text=rel["relative"].map("{:.2f}×".format),
                 hover_data={"relative": ":.2f", "likelihood": ":.1f", "average": ":.1f",
                             "log_rel": False, "level": False},
                 labels={"relative": "Times the average", "likelihood": "Estimated (%)",
                         "average": "Average (%)", "condition": "", "level": ""})
    lim = max(1.0, float(np.abs(rel["log_rel"]).max()) + 0.3)
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_layout(showlegend=False, yaxis_title=None, xaxis_title="Times the average (log scale)",
                      xaxis=dict(range=[-lim, lim], tickvals=[-2, -1, 0, 1, 2],
                                 ticktext=["¼×", "½×", "1× (average)", "2×", "4×"],
                                 zeroline=True, zerolinecolor=NAVY, zerolinewidth=2))
    return fig


def results_table(res):
    """Full results as a tidy table."""
    t = res.sort_values("likelihood", ascending=False)
    return pd.DataFrame({
        "Condition": t["condition"], "Estimated likelihood (%)": t["likelihood"].round(1),
        "Roughly": t["likelihood"].map(one_in), "Average, same age & sex (%)": t["average"].round(1),
        "Times the average": t["relative"].round(2), "Compared with average": t["level"],
        "Language / English effect": t["language_effect"].round(2),
        "Data source": np.where(t["own_data"], "This group", "Wider region")})

### 6.3 Try the estimator in the notebook

In [ ]:
P = dict(age="45–64", sex="Female", region=ANY, country="India", years="More than 10 years",
         language="Hindi", proficiency="Not well or Not at all")      # example profile – edit and re-run
res, info = estimate_profile(MIG, LANG, **P)
spec = res[res["condition"] != ANY_LTHC]
print(describe(P["age"], P["sex"], P["region"], P["country"], P["years"], P["language"], P["proficiency"]))
print(info)
fig_likelihood(spec).show()
fig_relative(spec).show()
results_table(res)

### 6.4 Risk Identifier app layout

In [ ]:
NOT_SPECIFIED = {"label": "Not specified", "value": ANY}


def opts(values):
    return [NOT_SPECIFIED] + [{"label": v, "value": v} for v in values]


def field(label, component, hint=None):
    kids = [html.Label(label, className="control-label"), component]
    if hint:
        kids.append(html.Div(hint, className="hint"))
    return html.Div(kids, className="form-field")


def step(num, title, *fields):
    return html.Div([html.Div([html.Span(str(num), className="step-num"), html.Span(title)],
                              className="step-title"), *fields], className="step")


def condition_card(row, rank):
    """Card for one of the top-3 conditions."""
    cls, rel = LEVEL_CLASS[row["level"]], row["relative"]
    rel_txt = (f"{rel:.1f}× the average" if pd.notna(rel) and rel >= 1 else
               f"{(1 - rel) * 100:.0f}% below average" if pd.notna(rel) else "")
    return html.Div([
        html.Div(f"#{rank}", className="rank"), html.Div(row["condition"], className="label"),
        html.Div(fmt_pct(row["likelihood"]), className="value"),
        html.Div(one_in(row["likelihood"]), className="sub"),
        html.Div([html.Span(row["level"], className=f"badge {cls}"), html.Span(rel_txt, className="rel")],
                 className="badge-row"),
    ], className=f"kpi cond-card {cls}")


risk_app = Dash("lthc_risk", title="LTHC Risk Identifier")
risk_app.index_string = page_template(CSS)

form = html.Div([
    html.H3("Tell us about the person"),
    html.P("Only age group and sex are required. Leave anything else as ‘Not specified’.", className="caption"),
    step(1, "About them",
         field("Age group *", dcc.RadioItems(id="p-age", options=AGE_ORDER, value=None, className="radio-stack")),
         field("Sex *", dcc.RadioItems(id="p-sex", value=None, className="radio-stack", options=[
             {"label": "Female", "value": "Female"}, {"label": "Male", "value": "Male"},
             {"label": "Not specified", "value": "Persons"}]))),
    step(2, "Where they were born",
         field("Region of birth", dcc.Dropdown(id="p-region", options=opts(REGIONS), value=ANY, clearable=False)),
         field("Country of birth", dcc.Dropdown(id="p-country", options=[NOT_SPECIFIED], value=ANY,
                                                clearable=False), hint="Choosing a region shortens this list."),
         field("Years living in Australia", dcc.RadioItems(id="p-years", value=ANY, className="radio-stack", options=[
             {"label": "Up to 10 years", "value": "0–10 years"},
             {"label": "More than 10 years", "value": "More than 10 years"}, NOT_SPECIFIED]))),
    step(3, "Language",
         field("Language used at home", dcc.Dropdown(id="p-language", options=[NOT_SPECIFIED], value=ANY,
                                                     clearable=False),
               hint="Languages recorded for the chosen country / region."),
         field("How well do they speak English?", dcc.RadioItems(id="p-prof", value=ANY, className="radio-stack",
               options=[{"label": "Very well or well", "value": "Very well or well"},
                        {"label": "Not well or not at all", "value": "Not well or Not at all"}, NOT_SPECIFIED]))),
    html.Div(id="form-error", className="form-error"),
    html.Button("Show likely conditions", id="submit", n_clicks=0, className="btn-primary"),
], className="card form-card")

placeholder = html.Div([
    html.Div("?", className="placeholder-icon"),
    html.H3("Your results will appear here"),
    html.P("Choose an age group and sex (and anything else you know), then click “Show likely conditions”."),
], className="card placeholder")

risk_app.layout = html.Div([
    html.Div([
        html.Div([html.H1("Long-Term Health Condition Risk Identifier"),
                  html.P("Fill in the form to see which long-term health conditions people with a similar "
                         "background are most likely to report (Census 2021, overseas-born Australians).")]),
        html.A("Open analysis dashboard →", href="http://127.0.0.1:8050", target="_blank", className="header-link"),
    ], className="header header-flex"),
    html.Div([form, dcc.Loading(html.Div(placeholder, id="results"), type="circle", color=TEAL)],
             className="page form-layout"),
    html.Div("Population-level estimates from aggregated census counts – not a diagnosis or a personal "
             "medical prediction. Source: AIHW, Census 2021.", className="footer"),
])

### 6.5 Risk Identifier callbacks

In [ ]:
@risk_app.callback(Output("p-country", "options"), Output("p-country", "value"),
                   Input("p-region", "value"), State("p-country", "value"))
def update_country_options(region, country):
    """Only list countries in the chosen region."""
    src = MIG if region == ANY else MIG[MIG["region"] == region]
    countries = sorted(src.loc[src["population"] > 0, "country"].unique())
    return opts(countries), country if country in countries else ANY


@risk_app.callback(Output("p-language", "options"), Output("p-language", "value"),
                   Input("p-region", "value"), Input("p-country", "value"), State("p-language", "value"))
def update_language_options(region, country, language):
    """Only list languages recorded for the chosen country / region."""
    src = LANG
    if country != ANY:
        src = src[src["country"] == country]
    elif region != ANY:
        src = src[src["region"] == region]
    langs = sorted(src.loc[src["population"] > 0, "language"].unique())
    return opts(langs), language if language in langs else ANY


@risk_app.callback(Output("results", "children"), Output("form-error", "children"),
                   Input("submit", "n_clicks"),
                   State("p-region", "value"), State("p-country", "value"), State("p-language", "value"),
                   State("p-age", "value"), State("p-sex", "value"), State("p-years", "value"),
                   State("p-prof", "value"), prevent_initial_call=True)
def show_results(_, region, country, language, age, sex, years, prof):
    """Run the estimator and build the results panel."""
    missing = [name for name, v in [("age group", age), ("sex", sex)] if not v]
    if missing:
        return placeholder, f"Please choose {' and '.join(missing)}."

    res, info = estimate_profile(MIG, LANG, age=age, sex=sex, region=region, country=country,
                                 years=years, language=language, proficiency=prof)
    any_row = res[res["condition"] == ANY_LTHC].iloc[0]
    spec = res[res["condition"] != ANY_LTHC].sort_values("likelihood", ascending=False)
    conf_cls = {"High": "Lower", "Medium": "Typical", "Low": "Elevated"}[info["confidence"]]
    group_n = info.get("language_population") if (language != ANY or prof != ANY) else info["population"]

    headline = html.Div([
        html.Div([
            html.Div("Profile", className="label"),
            html.Div(describe(age, sex, region, country, years, language, prof), className="profile-text"),
            html.Div([html.Span(f"{info['confidence']} confidence", className=f"badge {conf_cls}"),
                      html.Span(f"based on a census group of {fmt_int(group_n or 0)} people", className="rel")]
                     + [html.Div(n, className="warn") for n in info["notes"]], className="badge-row"),
        ], className="headline-left"),
        html.Div([
            html.Div("Likelihood of at least one long-term condition", className="label"),
            html.Div(fmt_pct(any_row["likelihood"], 0), className="big"),
            html.Div(f"{one_in(any_row['likelihood'])} · average {fmt_pct(any_row['average'], 0)}", className="sub"),
        ], className="headline-right"),
    ], className="card headline")

    top3 = html.Div([html.Div("Most likely conditions", className="section-title"),
                     html.Div([condition_card(r, i + 1) for i, (_, r) in enumerate(spec.head(3).iterrows())],
                              className="top-grid")])

    charts = html.Div([
        card("Estimated likelihood of each condition", "Bars = estimated % of people with this profile. "
             "◆ = average for all overseas-born people of the same age group and sex.",
             dcc.Graph(figure=fig_likelihood(spec), config=GRAPH_CFG, style={"height": "460px"})),
        card("Compared with the average", "How many times more (right) or less (left) likely than the "
             "average person of the same age and sex.",
             dcc.Graph(figure=fig_relative(spec), config=GRAPH_CFG, style={"height": "460px"})),
    ], className="grid-2")

    t = results_table(res)
    table = card("Full results", "Click a column header to sort, or Export to download as CSV.",
                 dash_table.DataTable(data=t.to_dict("records"), columns=table_columns(t), sort_action="native",
                                      page_size=12, export_format="csv", export_headers="display", **TABLE_STYLE))
    return [headline, top3, charts, table], ""

### 6.6 Show the Risk Identifier
Fill in the form and click **Show likely conditions**. Change `"inline"` to `"external"` to open it at http://127.0.0.1:8052 instead.

In [ ]:
risk_app.run(jupyter_mode="inline", jupyter_height=1300, port=8052, debug=False)